In [57]:
import pandas as pd
import numpy as np
import os
import polars as pl

# Load Yelp data from raw_data directory
raw_data_path = './Yelp_JSON_2018/'
target_path = './dataset2018/'
filter_size = 5

In [58]:
import ijson
f = open(os.path.join(raw_data_path, 'yelp_academic_dataset_review.json'), 'rb')
parser = ijson.parse(f, multiple_values = True)

In [59]:
import csv
with open(os.path.join(raw_data_path, 'yelp_review_reduced.csv'), "w") as csv_file:
        writer = csv.writer(csv_file, delimiter=' ')
        data = []
        for prefix, event, value in parser:
            if prefix=="review_id":
                data.append(value)
            elif prefix=="user_id":
                data.append(value)
            elif prefix=="business_id":
                data.append(value)
            elif prefix=="date":
                data.append(value)
            elif event == "end_map":
                writer.writerow(data)
                data = []

In [60]:
review_df = pd.read_csv(os.path.join(raw_data_path, 'yelp_review_reduced.csv'), sep = ' ', header=None)

In [61]:
review_df = review_df.rename(columns={0:'review_id', 1:'user_id', 2:'business_id', 3:'date'})

In [62]:
# Create business_id to incremental id mapping
unique_business_ids = review_df['business_id'].unique()
business_id_to_idx = {business_id: idx for idx, business_id in enumerate(unique_business_ids)}

# Create user_id to incremental id mapping
unique_user_ids = review_df['user_id'].unique()
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(unique_user_ids)}


In [63]:
# Filter items by popularity
item_counts = review_df['business_id'].value_counts()
popular_items = item_counts[item_counts >= filter_size].index.tolist()

# Filter reviews to only include popular items
filtered_reviews = review_df[review_df['business_id'].isin(popular_items)]

# Filter users by activity
user_counts = filtered_reviews['user_id'].value_counts()
active_users = user_counts[user_counts >= filter_size].index.tolist()

# Filter reviews to only include active users
filtered_reviews = filtered_reviews[filtered_reviews['user_id'].isin(active_users)]

# Get unique user IDs and shuffle them
user_ids = filtered_reviews['user_id'].unique()
np.random.shuffle(user_ids)

# Split users into train/valid/test
num_users = len(user_ids)
split_1 = int(num_users * 0.8)
split_2 = int(num_users * 0.9)
train_users = user_ids[:split_1]
valid_users = user_ids[split_1:split_2]
test_users = user_ids[split_2:]

# Process each split
train_data = filtered_reviews[filtered_reviews['user_id'].isin(train_users)]
valid_data = filtered_reviews[filtered_reviews['user_id'].isin(valid_users)]
test_data = filtered_reviews[filtered_reviews['user_id'].isin(test_users)]

In [64]:
train_data.shape[0]

2714195

In [65]:
from tqdm import tqdm
# Process validation data
valid_formatted = pd.DataFrame(columns=['user_id', 'given_user_history', 'predicting_items'])

for user_id, group in tqdm(valid_data.groupby('user_id')):
    # Sort by date first
    sorted_group = group.sort_values('date')
    business_ids = sorted_group['business_id'].tolist()
    split_idx = int(len(business_ids) * 0.8)
    given_history = [business_id_to_idx[bid] for bid in business_ids[:split_idx]]
    predicting_items = [business_id_to_idx[bid] for bid in business_ids[split_idx:]]
    
    
    new_row = pd.DataFrame({
        'user_id': [user_id_to_idx[user_id]],
        'given_user_history': [given_history],
        'predicting_items': [predicting_items]
    })
    valid_formatted = pd.concat([valid_formatted, new_row], ignore_index=True)

valid_formatted.to_csv(os.path.join(target_path, 'validation.tsv'), sep='\t', index=False)


100%|██████████| 22100/22100 [00:20<00:00, 1054.96it/s]


In [66]:

# Process test data
test_formatted = pd.DataFrame(columns=['user_id', 'given_user_history', 'predicting_items'])

for user_id, group in tqdm(test_data.groupby('user_id')):
    # Sort by date first
    sorted_group = group.sort_values('date')
    business_ids = sorted_group['business_id'].tolist()
    split_idx = int(len(business_ids) * 0.8)
    given_history = [business_id_to_idx[bid] for bid in business_ids[:split_idx]]
    predicting_items = [business_id_to_idx[bid] for bid in business_ids[split_idx:]]
    
    new_row = pd.DataFrame({
        'user_id': [user_id_to_idx[user_id]],
        'given_user_history': [given_history],
        'predicting_items': [predicting_items]
    })
    test_formatted = pd.concat([test_formatted, new_row], ignore_index=True)

test_formatted.to_csv(os.path.join(target_path, 'test.tsv'), sep='\t', index=False)


100%|██████████| 22100/22100 [00:21<00:00, 1018.01it/s]


In [67]:
import csv
# Process training data
train_formatted = pd.DataFrame(columns=['user_id', 'given_user_history', 'predicting_items'])

with open(os.path.join(target_path, 'training.tsv'), "w") as csv_file:
    writer = csv.writer(csv_file, delimiter="\t")
    writer.writerow(["user_id", "given_user_history", "predicting_items"])
    for user_id, group in tqdm(train_data.groupby('user_id')):
        # Sort by date first
        sorted_group = group.sort_values('date')
        business_ids = sorted_group['business_id'].tolist()
        
        # Create all possible sequences
        for i in range(1, len(business_ids)):
            history = [business_id_to_idx[bid] for bid in business_ids[:i]]
            target = business_id_to_idx[business_ids[i]]
            
            writer.writerow([user_id_to_idx[user_id], history, target])
#             new_row = pd.DataFrame({
#                 'uid': [user_id_to_idx[user_id]],
#                 'his_seq': [history],
#                 'next_item': [target]
#             })
#             train_formatted = pd.concat([train_formatted, new_row], ignore_index=True)

# train_formatted.to_csv(os.path.join(target_path, 'training.tsv'), sep='\t', index=False)


100%|██████████| 176799/176799 [01:32<00:00, 1901.50it/s]


In [68]:
df = pd.read_csv('./dataset2018/training.tsv', sep='\t')
print(df['user_id'].nunique())


df = pd.read_csv('./dataset2018/validation.tsv', sep='\t')
print(df['user_id'].nunique())


df = pd.read_csv('./dataset2018/test.tsv', sep='\t')
print(df['user_id'].nunique())

176799
22100
22100


In [69]:
df = pd.read_csv('./dataset2018/training.tsv', sep='\t')
print(df['user_id'].nunique())

176799


In [70]:
print("Training DataFrame:")
display(df.head())

Training DataFrame:


,user_id,given_user_history,predicting_items
0,919827,[9427],388
1,919827,"[9427, 388]",18793
2,919827,"[9427, 388, 18793]",21349
3,919827,"[9427, 388, 18793, 21349]",18947
4,919827,"[9427, 388, 18793, 21349, 18947]",8159


In [71]:
df = pd.read_csv('./dataset2018/test.tsv', sep='\t')
print(df.head())

   user_id                                 given_user_history   
0  1025004                   [91605, 9900, 54990, 5177, 5189]  \
1  1304890                      [15455, 12109, 127309, 82947]   
2    74014           [32057, 75430, 1284, 14480, 1044, 19561]   
3   652099                         [6471, 8195, 61466, 17698]   
4   274571  [2184, 7457, 70322, 2632, 6017, 9316, 12645, 4...   

                          predicting_items  
0                             [5477, 5206]  
1                            [25783, 8369]  
2                            [1387, 17208]  
3                            [18541, 2274]  
4  [78112, 2371, 40323, 26418, 7006, 4255]  
